# Progressive Growing GANs (ProGAN)

## Theory, mathematics, and implementation context

A GAN learns through a two-player game. $G_\theta(z)$ maps noise $z\sim p(z)$ to images and $D_\phi(x)$ scores realism. The minimax objective is $\min_G\max_D\;\mathbb E_{p_{data}}\log D(x)+\mathbb E_z\log(1-D(G(z)))$; the non-saturating generator loss $-\mathbb E_z\log D(G(z))$ usually gives stronger early gradients. ProGAN grows both networks over spatial scales, for example 4→8→16→32 pixels. At each transition, old and new paths blend as $y=(1-\alpha)y_{old}+\alpha y_{new}$ while $\alpha$ ramps from zero to one. This continuation schedule lets coarse structure form before fine detail is introduced.

ProGAN also popularized equalized learning-rate scaling, pixel-wise feature normalization, and minibatch standard deviation. These address signal scale, feature magnitude, and batch-level sample diversity respectively. This notebook implements the ideas on 4,096 CIFAR-10 images at 32×32: a compact teaching model, not a reproduction of the original high-resolution WGAN-GP recipe. Full convergence takes many updates.

## Case study: reproducible small data slice

Images are normalized to [-1,1] for tanh outputs. CIFAR-10 download is handled by torchvision. A fixed subset keeps the comparison affordable.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, utils
import matplotlib.pyplot as plt
torch.manual_seed(7)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
tf=transforms.Compose([transforms.Resize(32),transforms.ToTensor(),transforms.Normalize((.5,)*3,(.5,)*3)])
ds=datasets.CIFAR10("./data",train=True,download=True,transform=tf)
loader=DataLoader(Subset(ds,range(4096)),batch_size=64,shuffle=True,num_workers=0,drop_last=True)
print(DEVICE,len(loader.dataset))

## Equalized learning rate and PixelNorm

Equalized learning rate applies fan-in scaling at runtime. PixelNorm normalizes feature channels independently at every spatial position.

In [ ]:
class EqConv(nn.Module):
    def __init__(self,ci,co,k,p=0):
        super().__init__(); self.weight=nn.Parameter(torch.randn(co,ci,k,k)); self.bias=nn.Parameter(torch.zeros(co)); self.scale=(ci*k*k)**-.5; self.p=p
    def forward(self,x): return F.conv2d(x,self.weight*self.scale,self.bias,padding=self.p)
class PixelNorm(nn.Module):
    def forward(self,x): return x*torch.rsqrt(x.square().mean(1,keepdim=True)+1e-8)
class Block(nn.Module):
    def __init__(self,ci,co):
        super().__init__(); self.net=nn.Sequential(EqConv(ci,co,3,1),nn.LeakyReLU(.2),PixelNorm(),EqConv(co,co,3,1),nn.LeakyReLU(.2),PixelNorm())
    def forward(self,x): return self.net(x)
CHANNELS={4:256,8:128,16:64,32:32}

## Generator with fade-in transition

ToRGB projections expose every stage. During transition, the old RGB image is upsampled and mixed with the new projection.

In [ ]:
class ProGenerator(nn.Module):
    def __init__(self,zdim=128):
        super().__init__(); self.start=nn.Sequential(nn.ConvTranspose2d(zdim,256,4),nn.LeakyReLU(.2),EqConv(256,256,3,1),nn.LeakyReLU(.2),PixelNorm())
        self.blocks=nn.ModuleList([Block(CHANNELS[r//2],CHANNELS[r]) for r in (8,16,32)])
        self.rgb=nn.ModuleList([nn.Conv2d(CHANNELS[r],3,1) for r in (4,8,16,32)])
    def forward(self,z,stage=3,alpha=1.):
        x=self.start(z)
        for i in range(stage):
            x=self.blocks[i](F.interpolate(x,scale_factor=2,mode="nearest"))
            if i==stage-1 and alpha<1:
                new=self.rgb[i+1](x); prev=F.interpolate(self.rgb[i](F.avg_pool2d(x,2)),scale_factor=2,mode="nearest")
                return torch.tanh(alpha*new+(1-alpha)*prev)
        return torch.tanh(self.rgb[stage](x))
G=ProGenerator().to(DEVICE); print(G(torch.randn(2,128,1,1,device=DEVICE),stage=3,alpha=.5).shape)

## Discriminator with minibatch statistic

The minibatch statistic adds a channel summarizing feature variation across examples. This compact discriminator uses logistic loss; the original ProGAN used WGAN-GP.

In [ ]:
class ProDiscriminator(nn.Module):
    def __init__(self):
        super().__init__(); self.fromrgb=nn.ModuleList([nn.Conv2d(3,CHANNELS[r],1) for r in (4,8,16,32)])
        self.down=nn.ModuleList([Block(CHANNELS[r],CHANNELS[r//2]) for r in (32,16,8)])
        self.final=nn.Sequential(EqConv(257,256,3,1),nn.LeakyReLU(.2),nn.Flatten(),nn.Linear(256*4*4,1))
    def forward(self,x,stage=3):
        y=F.leaky_relu(self.fromrgb[stage](x),.2); r=(4,8,16,32)[stage]
        while r>4:
            i={32:0,16:1,8:2}[r]; y=F.avg_pool2d(self.down[i](y),2); r//=2
        stat=(y.var(0,unbiased=False)+1e-8).sqrt().mean().view(1,1,1,1).expand(y.size(0),1,4,4)
        return self.final(torch.cat([y,stat],1))
D=ProDiscriminator().to(DEVICE); print(D(torch.randn(2,3,32,32,device=DEVICE)).shape)

## Transition schedule and one adversarial update

Track progress by images seen, not epochs. A full run trains each stage during fade-in and stabilization before increasing resolution.

In [ ]:
def alpha_from_images(images_seen,transition_images): return min(1.,images_seen/max(1,transition_images))
print([alpha_from_images(n,10000) for n in (0,2500,5000,10000)])
G=ProGenerator().to(DEVICE); D=ProDiscriminator().to(DEVICE)
optG=torch.optim.Adam(G.parameters(),lr=1e-4,betas=(0,.99)); optD=torch.optim.Adam(D.parameters(),lr=1e-4,betas=(0,.99))
real,_=next(iter(loader)); real=real.to(DEVICE); b=len(real); ones=torch.ones(b,1,device=DEVICE); zeros=torch.zeros_like(ones)
fake=G(torch.randn(b,128,1,1,device=DEVICE)).detach(); ld=F.binary_cross_entropy_with_logits(D(real),ones)+F.binary_cross_entropy_with_logits(D(fake),zeros)
optD.zero_grad(); ld.backward(); optD.step()
lg=F.binary_cross_entropy_with_logits(D(G(torch.randn(b,128,1,1,device=DEVICE))),ones); optG.zero_grad(); lg.backward(); optG.step()
print(float(ld),float(lg))

## Inspect samples and limitations

Compare fixed latent vectors across stages. A healthy fade preserves composition as detail appears. Progressive growth is a curriculum, not a guarantee against mode collapse or poor coverage. Later StyleGAN work abandoned progressive growing while retaining carefully engineered synthesis and regularization.

In [ ]:
G.eval()
with torch.no_grad(): samples=G(torch.randn(16,128,1,1,device=DEVICE),stage=3).cpu()
plt.figure(figsize=(6,6)); plt.axis("off"); plt.imshow(utils.make_grid(samples,nrow=4,normalize=True).permute(1,2,0));

## Key Takeaways

- Progressive growing adds capacity in resolution stages; fade-in blending makes transitions continuous.
- Equalized learning rate, PixelNorm, and minibatch standard deviation address distinct training issues.
- Schedule transitions by examples seen.
- Original ProGAN used more extensive training and WGAN-GP.
- Higher resolution increases compute and data demands; progressive growth alone does not ensure diversity or semantic quality.